# Structural Preprocessing - MNI Registration 



**Objective :** register the subject's T1-weighted anatomical image to the MNI152 standard template using FSL FLIRT 

**Input :** skull-stripped T1 image from task 1 of preprocessing 

**Output :** registered T1 image, affine transformation matrix, before / after alignment visualizations  

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

import numpy as np
import nibabel as nib
from nilearn import plotting

# The notebook is located at the root of NX421-MiniProject-1.
from pathlib import Path

ROOT = Path("/data/Desktop/myfiles/NX421-MiniProject-1")

print("Repository exists:", ROOT.is_dir())
print("Repository:", ROOT)

# Original anatomical image provided for the mini-project
RAW_T1 = ROOT / "data" / "T1" / "T1w.nii.gz"

print("Working directory:", ROOT)
print("Original T1 exists:", RAW_T1.is_file())

if not RAW_T1.is_file():
    raise FileNotFoundError(
        "Cannot find data/T1w/T1w.nii.gz. "
        "Check your notebook's working directory."
    )

# Read only the image header to check the file
raw_img = nib.load(str(RAW_T1))

print("T1 shape:", raw_img.shape)
print("T1 voxel size:", raw_img.header.get_zooms()[:3])

# Configuration, paths and parameters 

### Inputs :
- Moving image (source) : skull-stripped subject T1
- Fixed image (reference) : MNI152 T1 1 mm brain template provided by FSL 

As a result of registration, we will have an estimate of an affine transformation that maps the subject's anatomy to the reference space

In [ ]:

# Source : INPUT FROM Yasmine's part 

# TO CHANGE LATER
# Replace the path with the output of Yasmine's code.

BRAIN_T1 = ROOT / "outputs" / "structural" / "T1w_brain.nii.gz"



# MNI152 REFERENCE
# Default is 1 mm resolution, which is appropriate for T1 anatomical registration 
# we can use another MNI resolution (eg. : 2 mm)

fsl_dir = os.environ.get("FSLDIR")

MNI_TEMPLATE = (
    Path(fsl_dir) / "data" / "standard" /
    "MNI152_T1_1mm_brain.nii.gz"
    if fsl_dir else None
)


# OUTPUTS OF THE REGISTRATION
# we can change filenames

OUT_DIR = ROOT / "outputs" / "registration_mni"

REGISTERED_T1 = OUT_DIR / "T1w_brain_MNI152.nii.gz"
AFFINE_MAT = OUT_DIR / "T1w_to_MNI152_FLIRT.mat"

BEFORE_PNG = OUT_DIR / "alignment_before.png"
AFTER_PNG = OUT_DIR / "alignment_after.png"


# REGISTRATION PARAMETERS

# 12 DOF allow translation, rotation scaling and shear 
# this is an initial choice to match images of the same modality 
# we can adjust it if we can justify (eg. : 6 DOF for rigid registration)
DOF = 12

# suitable cost for a T1-to-T1 alignment : normalized correlation
# we can also test "normmi" if we are not satisfied with the alignment
COST = "normcorr"

# Same MNI coordinates in the before/after plots to keep visualization coordinates consistent
# we can adjust if we want to display more informative anatomical slices 
CUT_COORDS = (0, -20, 20)
 

# CHECK FILE AVAILABILITY (to be removed when we put all structural preprocessing files together)

print("Skull-stripped T1 exists:", BRAIN_T1.is_file())
print("FLIRT available:", shutil.which("flirt") is not None)
print(
    "MNI template exists:",
    bool(MNI_TEMPLATE and MNI_TEMPLATE.is_file())
)

# Verifying registration inputs 

Before we run FLIRT, we need to verify that the subject's skull-stripped anatomical image and MNI152 reference are available. 

These 2 inputs are 3D images. Dimensions and voxel resolutions can be different. FLIRT will be responsible for resampling the moving image onto the refrence grid.

In [ ]:
# Check that the Yasmine's output is available
if not BRAIN_T1.is_file():
    raise FileNotFoundError(
        "Skull-stripped T1 not yet available. "
        "Update BRAIN_T1 when Part 1(a) is complete."
    )

# Check FSL installation / Neurodesk environment
if (
    shutil.which("flirt") is None
    or MNI_TEMPLATE is None
    or not MNI_TEMPLATE.is_file()
):
    raise RuntimeError(
        "FSL is not loaded or the MNI template was not found."
    )

# Load both anatomical images
# BRAIN_T1 is the result of first structural preprocessing part (skull-stripped T1)
subject_img = nib.load(str(BRAIN_T1))
mni_img = nib.load(str(MNI_TEMPLATE))

assert len(subject_img.shape) == 3
assert len(mni_img.shape) == 3

print("Subject shape:", subject_img.shape)
print("MNI shape:", mni_img.shape)

print("Subject voxel size:", subject_img.header.get_zooms()[:3])
print("MNI voxel size:", mni_img.header.get_zooms()[:3])

# Create output directory if necessary
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Alignment before registration 

Before we estimate the transformation, we visually compare the subject's T1 with the MNI152 template. 

The template (reference image) is displayed as the background. The edges of the subject's anatomical image are superimposed without applying FLIRT. 

The goal of this step is to assess the initial spatial alignment. 

In [ ]:
# Displaying the MNI152 anatomical reference 
before = plotting.plot_anat(
    str(MNI_TEMPLATE),
    display_mode="ortho",
    cut_coords=CUT_COORDS,
    title="BEFORE registration: Subject T1 vs MNI152",
    draw_cross=False,
)

# Overlay anatomical edges from the subject's T1
before.add_edges(str(BRAIN_T1))

# This will give us 3 orthogonal views : sagittal, coronal, axial 
# Registration is not performed yet 
# This is the visualization that will be later useful to compare with the transformed image 

# Save figure
before.savefig(str(BEFORE_PNG))
before.close()

print("Figure saved:", BEFORE_PNG)

# Affine registration to MNI space 

We use FSL FLIRT to align the subject's skull-stripped T1 image (moving image) with the MNI152 brain template (fixed image). 

This affine transformation is optimized with a normalized correlation (DOF = 12).

The results (affine transformation matrix and registered image) are saved for the next analysis parts. 

In [ ]:
# Register the subject's T1 to MNI152 using FLIRT
# FLIRT estimates the transformation and resamples the T1 onto the MNI reference
# we used trilinear interpolation for continuous T1 images (as an initial choice)
# we can test other interpolation methods 
# the saved matrix contains a 4 x 4 affine transformation 
command = [
    "flirt",
    "-in", str(BRAIN_T1),
    "-ref", str(MNI_TEMPLATE),
    "-out", str(REGISTERED_T1),
    "-omat", str(AFFINE_MAT),
    "-dof", str(DOF),
    "-cost", COST,
    "-interp", "trilinear", # we can change interpolation method
]


print("Running FLIRT registration...")

# explicitly visualizing inputs, options and output matrix 
subprocess.run(command, check=True)


print("Registration completed.")
print("Registered image:", REGISTERED_T1)
print("Transformation matrix:", AFFINE_MAT)

# Quality control 

We overlay the registered image on the MNI152 reference to assess the quality of alignment. 

The same reference and cut coordinates are used for the before and after figures. 

We can visually evaluate the alignment quality using anatomical landmarks such as the ventricles, cortical boundaries, midline and cerebellum. 

In [ ]:
# Plotting MNI reference with registered subject T1 edges 
after = plotting.plot_anat(
    str(MNI_TEMPLATE),
    display_mode="ortho",
    cut_coords=CUT_COORDS,
    title="AFTER registration: Subject T1 vs MNI152",
    draw_cross=False,
)

after.add_edges(str(REGISTERED_T1))

after.savefig(str(AFTER_PNG))
after.close()

print("Figure saved:", AFTER_PNG)


# Check the registered image
# The grid check verifies that the output matches the spatial sampling of the reference, without proving registration quality. 
registered_img = nib.load(str(REGISTERED_T1))

print("Registered image shape:", registered_img.shape)
print("MNI template shape:", mni_img.shape)

print(
    "Output grid matches reference:",
    registered_img.shape == mni_img.shape
    and np.allclose(
        registered_img.affine,
        mni_img.affine,
        atol=1e-3
    )
)


# Display the saved transformation matrix
affine_matrix = np.loadtxt(AFFINE_MAT)

print("FLIRT transformation matrix:")
print(affine_matrix)

# Registration results 

The subjects's skull stripped T1-weighted anatomical image was registred to the MNI152 1 mm template using FSL FLIRT with a 12-DOF affine transformation and normalized-correlation cost. 

Quality control observations : TO BE COMPLETED when we take into consideration the first preprocessing part (yasmine) and analysing the alignment before and after registration (check if ventricles, cortical boundaries and other brain structures align closely with the template)